Step 1: load your bench numbers

In [1]:
import json

try:
    levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
except FileNotFoundError:
    levels = [
        {"concurrency": 1,  "tokens_per_s": 38.2,  "latency_p95_s": 0.9,  "errors": 0},
        {"concurrency": 2,  "tokens_per_s": 71.5,  "latency_p95_s": 1.1,  "errors": 0},
        {"concurrency": 4,  "tokens_per_s": 128.4, "latency_p95_s": 1.4,  "errors": 0},
        {"concurrency": 8,  "tokens_per_s": 210.7, "latency_p95_s": 2.3,  "errors": 0},
        {"concurrency": 16, "tokens_per_s": 224.9, "latency_p95_s": 5.8,  "errors": 0},
    ]
    print("using the sample bench_report -- swap in your own file for a real answer")

for L in levels:
    print(L)

{'concurrency': 1, 'tokens_per_s': 83.5, 'ttft_p50_s': 0.0539, 'ttft_p95_s': 0.0847, 'latency_p95_s': 1.5388, 'errors': 0, 'ok': 20, 'wall_s': 23.989}
{'concurrency': 2, 'tokens_per_s': 158.85, 'ttft_p50_s': 0.0543, 'ttft_p95_s': 0.0795, 'latency_p95_s': 1.6164, 'errors': 0, 'ok': 20, 'wall_s': 12.61}
{'concurrency': 4, 'tokens_per_s': 275.37, 'ttft_p50_s': 0.0647, 'ttft_p95_s': 0.1225, 'latency_p95_s': 1.77, 'errors': 0, 'ok': 20, 'wall_s': 7.274}
{'concurrency': 8, 'tokens_per_s': 463.05, 'ttft_p50_s': 0.1243, 'ttft_p95_s': 0.1718, 'latency_p95_s': 1.9972, 'errors': 0, 'ok': 20, 'wall_s': 4.326}
{'concurrency': 16, 'tokens_per_s': 723.84, 'ttft_p50_s': 0.248, 'ttft_p95_s': 0.2508, 'latency_p95_s': 2.4006, 'errors': 0, 'ok': 20, 'wall_s': 2.781}


Step 2: the cost formula

In [2]:
def cost_per_million_tokens(tokens_per_s, gpu_hourly_usd):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = tokens_per_hour / 1_000_000
    return round(gpu_hourly_usd / million_tokens_per_hour, 4)

GPU_HOURLY_USD = 0.35

for L in levels:
    L["cost_per_million_tokens_usd"] = cost_per_million_tokens(
        L["tokens_per_s"],
        GPU_HOURLY_USD
    )

for L in levels:
    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"p95={L['latency_p95_s']:.2f}s  "
        f"$/M tok=${L['cost_per_million_tokens_usd']}"
    )

c= 1  tok/s=   83.5  p95=1.54s  $/M tok=$1.1643
c= 2  tok/s=  158.8  p95=1.62s  $/M tok=$0.612
c= 4  tok/s=  275.4  p95=1.77s  $/M tok=$0.3531
c= 8  tok/s=  463.1  p95=2.00s  $/M tok=$0.21
c=16  tok/s=  723.8  p95=2.40s  $/M tok=$0.1343


Step 3: find the knee, price it, and name what's past it

In [3]:
TARGET_P95_S = 5.0

under_target = [
    L for L in levels
    if L["latency_p95_s"] <= TARGET_P95_S
]

knee = max(
    under_target,
    key=lambda L: L["concurrency"]
) if under_target else None

print("knee:", knee)

past_knee = [
    L for L in levels
    if knee and L["concurrency"] > knee["concurrency"]
]

if past_knee:
    cheapest_past_knee = min(
        past_knee,
        key=lambda L: L["cost_per_million_tokens_usd"]
    )
    print(
        "cheapest $/M token level past the knee (SLO-violating):",
        cheapest_past_knee
    )
    print(
        "-> cheaper on paper, but its p95 already exceeds your SLO."
    )
else:
    print("No measured level is past the knee; this sweep is sweep-bounded.")

knee: {'concurrency': 16, 'tokens_per_s': 723.84, 'ttft_p50_s': 0.248, 'ttft_p95_s': 0.2508, 'latency_p95_s': 2.4006, 'errors': 0, 'ok': 20, 'wall_s': 2.781, 'cost_per_million_tokens_usd': 0.1343}
No measured level is past the knee; this sweep is sweep-bounded.


Step 4: the scale-out model

In [4]:
import math

def replicas_needed(required_tokens_per_s, knee_tokens_per_s):
    return math.ceil(required_tokens_per_s / knee_tokens_per_s)

def scale_out_cost(required_tokens_per_s, knee, gpu_hourly_usd):
    n = replicas_needed(required_tokens_per_s, knee["tokens_per_s"])
    return {
        "required_tokens_per_s": required_tokens_per_s,
        "replicas_needed": n,
        "total_hourly_cost_usd": round(n * gpu_hourly_usd, 2),
        "effective_p95_s": knee["latency_p95_s"],
    }

targets = [
    knee["tokens_per_s"] * m
    for m in (1.0, 1.5, 2.0, 3.0)
]

scale_plan = [
    scale_out_cost(t, knee, GPU_HOURLY_USD)
    for t in targets
]

for row in scale_plan:
    print(row)

{'required_tokens_per_s': 723.84, 'replicas_needed': 1, 'total_hourly_cost_usd': 0.35, 'effective_p95_s': 2.4006}
{'required_tokens_per_s': 1085.76, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.4006}
{'required_tokens_per_s': 1447.68, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.4006}
{'required_tokens_per_s': 2171.52, 'replicas_needed': 3, 'total_hourly_cost_usd': 1.05, 'effective_p95_s': 2.4006}


Step 5: write cost_report.json


In [5]:
report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": levels,
    "knee": knee,
    "scale_out_plan": scale_plan,
}

with open("cost_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "gpu_hourly_usd": 0.35,
  "target_p95_s": 5.0,
  "levels": [
    {
      "concurrency": 1,
      "tokens_per_s": 83.5,
      "ttft_p50_s": 0.0539,
      "ttft_p95_s": 0.0847,
      "latency_p95_s": 1.5388,
      "errors": 0,
      "ok": 20,
      "wall_s": 23.989,
      "cost_per_million_tokens_usd": 1.1643
    },
    {
      "concurrency": 2,
      "tokens_per_s": 158.85,
      "ttft_p50_s": 0.0543,
      "ttft_p95_s": 0.0795,
      "latency_p95_s": 1.6164,
      "errors": 0,
      "ok": 20,
      "wall_s": 12.61,
      "cost_per_million_tokens_usd": 0.612
    },
    {
      "concurrency": 4,
      "tokens_per_s": 275.37,
      "ttft_p50_s": 0.0647,
      "ttft_p95_s": 0.1225,
      "latency_p95_s": 1.77,
      "errors": 0,
      "ok": 20,
      "wall_s": 7.274,
      "cost_per_million_tokens_usd": 0.3531
    },
    {
      "concurrency": 8,
      "tokens_per_s": 463.05,
      "ttft_p50_s": 0.1243,
      "ttft_p95_s": 0.1718,
      "latency_p95_s": 1.9972,
      "errors": 0,
     

# **Verify (green check)**


In [6]:
!python verify.py

recomputed costs, knee and scale-out plan all agree
GREEN CHECK: PASS
